# Chapter 7 · 기상 체제변화 강건성

**연구질문:** 기상 체제변화(평균 이동) 전후에 예측모형 성능이 안정적인가? 구조변화 검정은?

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**누수(leakage):** 예측 시점 $t$에 알 수 없는 정보가 특성·전처리·분할에 들어가 성능이 과대평가되는 문제. 대표 경로 세 가지.
1. **전처리 누수:** 표준화·결측대체·모수 추정을 전체 자료로 한다(학습구간 통계만 써야 함).
2. **미래 정보 누수:** 중심 이동평균 등 $y_{t+k}$를 포함하는 특성.
3. **분할 누수:** 시계열에 무작위 K-겹 교차검증을 쓴다(인접 관측이 학습·검증에 섞임).

**Walk-forward(롤링 오리진) 평가:** 시점 $T_1<T_2<\dots$마다 $[1,T_k]$로 학습하고 $(T_k,T_k+h]$를 평가한다. 폴드별 오차의 분포(평균·분산·최악)를 본다.

**Diebold–Mariano 검정:** 손실차 $d_t=L(e^{A}_t)-L(e^{B}_t)$의 평균이 0인지 검정한다. $\mathrm{DM}=\bar d/\sqrt{\widehat{\mathrm{LRV}}(d)/n}$.

**부트스트랩 신뢰구간:** 폴드 손실차를 블록 부트스트랩하여 우열의 불확실성을 본다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_openmeteo_daily(obj):
    d = obj["daily"]; idx = pd.to_datetime(d["time"])
    tmax = pd.Series(d["temperature_2m_max"], index=idx, dtype=float)
    tmin = pd.Series(d["temperature_2m_min"], index=idx, dtype=float)
    return ((tmax + tmin) / 2).rename("temp_mid")          # 최고·최저 평균(정의된 일평균 아님)

def parse_openmeteo_pm25(obj):
    h = obj["hourly"]; s = pd.Series(h["pm2_5"], index=pd.to_datetime(h["time"]), dtype=float)
    return s.resample("D").mean().rename("pm2_5")

def parse_daily_min_temp(raw):
    df = pd.read_csv(io.BytesIO(raw)); df.columns = [c.strip().lower() for c in df.columns]
    s = pd.Series(pd.to_numeric(df["temp"], errors="coerce").values, index=pd.to_datetime(df["date"]), name="temp_mid")
    if len(s) < 3000 or not s.dropna().between(-10, 40).all():
        raise ValueError("일 최저기온 길이/범위 이상")
    return s

def synthetic_temp(n=3652, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(12 + 13 * np.sin(2 * np.pi * (t - 110) / 365.25) + rng.normal(0, 3, n) + 0.0004 * t,
                     index=pd.date_range("2015-01-01", periods=n, freq="D"), name="temp_mid")

def load_energy(which="temp"):
    temp, pm, kind = None, None, "SYNTHETIC (NOT real data)"
    if USE_LIVE_DATA:
        try:
            u = ("https://archive-api.open-meteo.com/v1/archive?latitude=37.57&longitude=126.98"
                 "&start_date=2015-01-01&end_date=2024-12-31&daily=temperature_2m_max,temperature_2m_min&timezone=Asia%2FSeoul")
            temp = parse_openmeteo_daily(json.loads(fetch_bytes(u, timeout=120)))
            kind = "REAL: Open-Meteo/ERA5 재분석 서울 일 기온 (관측소 실측 아님)"
        except Exception as e:
            print("Open-Meteo 기온 수집 실패 -> 2순위 시도:", type(e).__name__, str(e)[:120])
            try:
                temp = parse_daily_min_temp(fetch_bytes("https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv", timeout=60))
                kind = "REAL: Melbourne 일 최저기온 1981-1990 (jbrownlee/Datasets; 서울 아님, 일평균 아닌 최저기온)"
            except Exception as e2:
                print("일 최저기온 수집 실패 -> 합성 대체:", type(e2).__name__, str(e2)[:100]); temp = None
        try:
            u = ("https://air-quality-api.open-meteo.com/v1/air-quality?latitude=37.57&longitude=126.98"
                 "&hourly=pm2_5&start_date=2022-09-01&end_date=2024-12-31&timezone=Asia%2FSeoul")
            pm = parse_openmeteo_pm25(json.loads(fetch_bytes(u, timeout=120)))
        except Exception as e:
            print("Open-Meteo 대기질 수집 실패(선택 항목):", type(e).__name__, str(e)[:100])
    if temp is None:
        temp = synthetic_temp()
    temp = temp.asfreq("D").interpolate()
    if which == "demand":
        rng = np.random.default_rng(SEED + 5); t = np.arange(len(temp))
        dow = temp.index.dayofweek.to_numpy()
        dem = 60 + 0.9 * np.abs(temp.to_numpy() - 18) ** 1.15 + 6 * (dow < 5) + 0.004 * t + rng.normal(0, 2.0, len(temp))
        return dict(y=pd.Series(dem, index=temp.index, name="demand"), X=temp.to_frame("temp"), m=7, freq="D",
                    label="일별 전력수요(합성, 기온 의존)", unit="GWh",
                    kind="SYNTHETIC-DERIVED (NOT real data): 수요는 합성, 기온 입력은 " + kind)
    return dict(y=temp, pm=pm, m=365, freq="D", label="일평균 기온", unit="°C", kind=kind)

D = load_energy("temp")
y, DATA_KIND, FREQ = D["y"], D["kind"], D["freq"]
M = D["m"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 체제변화 탐지와 예측 성능

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def make_X(s, lags=(1, 2, 3, 7, 14), rolls=(7, 28)):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rm_{w}"] = s.shift(1).rolling(w).mean()
    return X

def rolling_origin(s, n_folds=8, fold=None, kind="ridge", window=None, lags=(1, 2, 3, 7, 14)):
    s = s.astype(float); X = make_X(s, lags=lags); data = pd.concat([X, s.rename("target")], axis=1).dropna()
    XX, yy = data.drop(columns="target"), data["target"]; n = len(data); fold = fold or max(30, n // (n_folds + 3))
    start = n - n_folds * fold; out = []
    for k in range(n_folds):
        a, b = start + k * fold, start + (k + 1) * fold
        lo = 0 if window is None else max(0, a - window)
        mdl = make_pipeline(StandardScaler(), Ridge(alpha=1.0)) if kind == "ridge" else HistGradientBoostingRegressor(max_iter=120, learning_rate=0.06, random_state=SEED)
        mdl.fit(XX.iloc[lo:a], yy.iloc[lo:a]); p = mdl.predict(XX.iloc[a:b]); naive = XX["lag_1"].iloc[a:b].to_numpy()
        out.append({"fold": k + 1, "end": XX.index[b - 1], "RMSE": rmse(yy.iloc[a:b], p), "RMSE_naive": rmse(yy.iloc[a:b], naive)})
    return pd.DataFrame(out)

In [ ]:
# ---- 체제변화: 평균 이동을 주입한 기온 시계열과 원 시계열 비교 ----
s_all = y.astype(float)
brk = int(len(s_all) * 0.6); s_shift = s_all.copy(); s_shift.iloc[brk:] += 2.0          # 6번째 10분위 이후 평균 +2도(주입)
def sup_f(series, trim=0.15, step=30):
    # 평균 이동에 대한 sup-F(Chow) 스캔. 임계값은 비표준(Andrews)이므로 최댓값과 위치만 보고 "확인용"으로 쓴다.
    v = series.to_numpy(float); n = len(v); best = (0, None)
    for k in range(int(trim * n), int((1 - trim) * n), step):
        a, b = v[:k], v[k:]
        rss1 = np.sum((a - a.mean()) ** 2) + np.sum((b - b.mean()) ** 2); rss0 = np.sum((v - v.mean()) ** 2)
        F = (rss0 - rss1) / (rss1 / (n - 2))
        if F > best[0]: best = (F, k)
    return best
for name, ser in (("원 시계열", s_all), ("평균 이동 주입", s_shift)):
    # 계절성을 제거한 뒤 이동 검정(계절 평균 제거)
    clim = ser.groupby(ser.index.dayofyear).transform("mean"); anom = ser - clim
    F, k = sup_f(anom); print(f"{name}: sup-F={F:.1f}, 추정 이동 위치={ser.index[k].date() if k else None}  (주입 위치 {s_all.index[brk].date()})")

In [ ]:
# ---- 체제변화 전후 예측 성능 (롤링 오리진, 폴드별) ----
for name, ser in (("원 시계열", s_all), ("평균 이동 주입", s_shift)):
    R_ = rolling_origin(ser, n_folds=8, fold=120, kind="hgb", lags=(1, 2, 3, 7, 14))
    R_["상대"] = R_["RMSE"] / R_["RMSE_naive"]
    print(f"[{name}]"); show(R_[["fold", "end", "RMSE", "RMSE_naive", "상대"]])
print("해석: 체제변화 직후 폴드에서 트리 모형은 학습 범위 밖 값을 예측하지 못해 성능이 급락할 수 있다(외삽 한계). 롤링 학습 창이나 변화점 탐지를 연결해 본다.")

## 3. 결과 해석 작성
1. 정상 설정과 누수 설정의 성능 차이를 표로 제시하고, **어떤 누수 경로가 가장 큰 왜곡을 만들었는지** 설명한다.
2. 폴드(또는 지역·기간) 간 변동을 보고 "평균은 좋지만 최악의 경우가 나쁘다"를 구분한다.
3. 결론(모형 우열, 효과 방향)이 설정 변경에도 유지되는가? 유지되지 않으면 "강건하다"가 아니라 "민감하다"로 쓴다.
4. 한계: 폴드 수, 이벤트 수, 합성자료의 한계, 위약검정이 모든 식별 가정을 증명하지는 않는다는 점.

**제출:** 정상/누수 비교표, 폴드별 결과, 강건성 표, 한계 단락. 누수 점검 목록(`validation/Leakage_Checklist.md`)과 대조하라.